# CPU testing and error analysis: HOG + SVM vs YOLO

This notebook runs both detectors **on the same CPU** and scores them with **the same evaluation code** (`methods/hog_svm/evaluate.py`), then analyses where they fail.

| Step | What it produces |
|---|---|
| 1. Predictions | Test predictions for HOG and YOLO, made on this CPU |
| 2. Timing | ms per image on 1 core and on all cores, same images for every method |
| 3. Accuracy | mAP@0.5, mAP@0.5:0.95, precision, recall, F1, count error; PR curves |
| 4. Missed birds | Detected vs labelled birds per image, worst images, error pictures |
| 5. Bird size | Recall for small, medium and large birds |
| 6. Brightness | Accuracy on dark, medium and bright images |
| 7. Simulated lighting (optional) | Accuracy when test images are darkened or brightened |

**Rules.** Nothing here is tuned on the test set. The models are already trained and frozen. HOG's operating threshold (0.6) was chosen on validation by the HOG team. YOLO's operating threshold is chosen here on **validation** using the same best-F1 rule, then applied to test. Brightness bins are defined from the **training** images.

Every table and figure is saved to `results/cpu_testing/`, which is git-ignored.

## 0. Setup

Run this from WSL inside the shared repo, with the repo's `.venv` active and Jupyter started from the repo root:

```bash
cd ~/comp9517_CDVN && source .venv/bin/activate && jupyter notebook
```

The notebook expects these files, none of which are in Git:

| File | Where it comes from |
|---|---|
| `dataset/images/{train,val,test}`, `dataset/labels/...` | Symlink to your downloaded dataset: `ln -s ~/comp9517/data/beach-bird dataset` |
| `results/hog_svm/model.npz` | The trained HOG model, from the HOG team |
| `models/yolo_stage2.pt` | Drive: `comp9517_bird/runs/stage2/weights/best.pt` (the final YOLO model) |
| `models/yolo_no_freeze.pt` (optional) | Drive: `comp9517_bird/runs/baseline_no_freeze/weights/best.pt`. Skipped if absent |

Close other heavy programs while timing runs, and keep the laptop plugged in.

In [ ]:
import os
import platform
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (ROOT / "methods" / "hog_svm").exists(), f"Start Jupyter inside the comp9517_CDVN repo (now at {ROOT})"
sys.path.insert(0, str(ROOT))
os.environ["PYTHONPATH"] = str(ROOT)  # so HOG's parallel worker processes can import the package

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from threadpoolctl import threadpool_limits
from ultralytics import YOLO

from methods.hog_svm import config as C
from methods.hog_svm.data import image_paths, iou_matrix, load_boxes, load_image, load_predictions, save_predictions
from methods.hog_svm.detect import detect, load_model, run_split
from methods.hog_svm.evaluate import average_precision

# ---- settings ---------------------------------------------------------------
YOLO_MODELS = {
    "YOLO26n 2-stage": ROOT / "models" / "yolo_stage2.pt",          # final YOLO model
    "YOLO26n no-freeze": ROOT / "models" / "yolo_no_freeze.pt",     # optional comparison run
}
MAIN_YOLO = "YOLO26n 2-stage"    # the team's chosen YOLO design (frozen backbone, then full fine-tune)
HOG = "HOG + SVM"
YOLO_CONF = 0.001                # keep (almost) every box so the full PR curve is scored
YOLO_IMGSZ = 640                 # same input size as training

RUN_HOG_TEST = True              # re-run HOG on test on this CPU (~240 x 7.7 s / cores). False = load saved predictions
TIMING_IMAGES = 30               # 1-core timing uses this many test images (HOG takes ~8 s each on 1 core)
RUN_SYNTHETIC_YOLO = True        # step 7 for YOLO (fast)
RUN_SYNTHETIC_HOG = False        # step 7 for HOG (slow: ~1 full HOG test run per lighting level)

OUT = ROOT / "results" / "cpu_testing"
(OUT / "figures").mkdir(parents=True, exist_ok=True)
IOU_THRESHOLDS = np.round(np.arange(0.5, 0.96, 0.05), 2)

# ---- checks -----------------------------------------------------------------
missing = [p for p in [C.DATA / "images" / "test", C.MODEL_PATH, YOLO_MODELS[MAIN_YOLO]] if not p.exists()]
assert not missing, "Missing files:\n" + "\n".join(map(str, missing))
for name in [n for n, p in YOLO_MODELS.items() if not p.exists()]:
    print(f"Skipping optional model '{name}': {YOLO_MODELS.pop(name)} not found")
for split, n in {"train": 1369, "val": 291, "test": 240}.items():
    print(f"{split}: {len(image_paths(split))} images (expected {n})")


def cpu_name():
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or "unknown CPU"


CPU = cpu_name()
CORES = os.cpu_count()
print(f"\nCPU: {CPU} | {CORES} logical cores | torch {torch.__version__} | Python {platform.python_version()}")
print("CUDA in use: no (device='cpu' everywhere)")

In [ ]:
# ---- shared helpers -----------------------------------------------------------

def ground_truth(split):
    return {p.stem: load_boxes(split, p.stem) for p in image_paths(split)}


def yolo_predict(model, sources, stems):
    # Predict on CPU. sources: image paths or BGR arrays. Returns ({stem: (boxes, scores)}, seconds per image).
    preds, seconds = {}, []
    for src, stem in zip(sources, stems):
        t = time.perf_counter()
        r = model.predict(src if not isinstance(src, Path) else str(src), conf=YOLO_CONF, imgsz=YOLO_IMGSZ,
                          max_det=300, device="cpu", verbose=False)[0]
        seconds.append(time.perf_counter() - t)
        preds[stem] = (r.boxes.xyxy.numpy().astype(np.float32), r.boxes.conf.numpy().astype(np.float32))
    return preds, np.array(seconds)


def warm_up(model):
    # The first call builds the network; exclude it from timing.
    model.predict(np.zeros((512, 512, 3), np.uint8), imgsz=YOLO_IMGSZ, device="cpu", verbose=False)


def match(boxes, scores, gt, iou_thr=0.5):
    # Same greedy matching as evaluate.py. Returns boxes sorted by score, a flag per box (true positive?)
    # and a flag per ground-truth box (found?).
    order = np.argsort(-scores)
    boxes = boxes[order]
    ious = iou_matrix(boxes, gt)
    found = np.zeros(len(gt), bool)
    tp = np.zeros(len(boxes), bool)
    for k in range(len(boxes)):
        if not len(gt):
            break
        j = int(np.argmax(np.where(found, -1, ious[k])))
        if not found[j] and ious[k, j] >= iou_thr:
            found[j] = tp[k] = True
    return boxes, tp, found


def above(preds, thr):
    return {s: (b[c > thr], c[c > thr]) for s, (b, c) in preds.items()}


def counts(preds, gt, thr, stems=None):
    tp = fp = fn = 0
    for s in (stems if stems is not None else gt):
        b, c = preds[s]
        _, t, f = match(b[c > thr], c[c > thr], gt[s])
        tp, fp, fn = tp + t.sum(), fp + (~t).sum(), fn + (~f).sum()
    return int(tp), int(fp), int(fn)


def best_f1_threshold(preds, gt):
    # Operating threshold = score with the best F1. Used on VALIDATION only.
    best = (0.0, None)
    for thr in np.round(np.arange(0.05, 0.96, 0.025), 3):
        tp, fp, fn = counts(preds, gt, thr)
        f1 = 2 * tp / max(2 * tp + fp + fn, 1)
        best = max(best, (f1, thr))
    return best[1], best[0]


def scores_table(preds, gt, thr, stems=None):
    stems = list(gt) if stems is None else stems
    p = {s: preds[s] for s in stems}
    g = {s: gt[s] for s in stems}
    n_birds = sum(len(b) for b in g.values())
    tp, fp, fn = counts(preds, gt, thr, stems)
    ap = {t: average_precision(p, g, t)[0] for t in IOU_THRESHOLDS} if n_birds else {}
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {
        "images": len(stems),
        "birds": n_birds,
        "mAP@0.5": ap.get(0.5, np.nan),
        "mAP@0.5:0.95": float(np.mean(list(ap.values()))) if ap else np.nan,
        "precision": prec if n_birds else np.nan,
        "recall": rec if n_birds else np.nan,
        "F1": (2 * prec * rec / (prec + rec) if prec + rec else 0.0) if n_birds else np.nan,
        "false boxes / image": fp / max(len(stems), 1),
        "count MAE": float(np.mean([abs(int((preds[s][1] > thr).sum()) - len(gt[s])) for s in stems])),
    }


def savefig(name):
    path = OUT / "figures" / name
    plt.savefig(path, dpi=200, bbox_inches="tight")
    print("saved", path.relative_to(ROOT))


GT = {"val": ground_truth("val"), "test": ground_truth("test")}
TEST_PATHS = image_paths("test")
TEST_STEMS = [p.stem for p in TEST_PATHS]
print({k: sum(len(b) for b in v.values()) for k, v in GT.items()}, "labelled birds (val, test)")

## 1. Predictions on this CPU

YOLO predicts on validation (only to choose its operating threshold) and on test. HOG is re-run on test with its frozen model and settings, so its time is measured on this machine too. Running inference again on a frozen model isn't tuning, so it doesn't break the test-only-once rule.

In [ ]:
PREDS, ALL_CORE_MS, OP_THR = {}, {}, {}
torch.set_num_threads(CORES)

for name, path in YOLO_MODELS.items():
    model = YOLO(str(path))
    warm_up(model)
    val_preds, _ = yolo_predict(model, image_paths("val"), list(GT["val"]))
    thr, f1 = best_f1_threshold(val_preds, GT["val"])
    OP_THR[name] = thr
    wall = time.perf_counter()
    PREDS[name], _ = yolo_predict(model, TEST_PATHS, TEST_STEMS)
    ALL_CORE_MS[name] = 1000 * (time.perf_counter() - wall) / len(TEST_PATHS)
    for stem, (b, c) in PREDS[name].items():
        save_predictions(OUT / "preds" / name.replace(" ", "_") / f"{stem}.txt", b, c)
    print(f"{name}: operating threshold {thr} (val F1 {f1:.3f}) | test {ALL_CORE_MS[name]:.1f} ms/img on {CORES} cores")

OP_THR[HOG] = C.OPERATING_THRESHOLD
if RUN_HOG_TEST:
    w, b = load_model()
    wall = time.perf_counter()
    PREDS[HOG], _ = run_split("test", w, b)           # parallel over images, all cores
    ALL_CORE_MS[HOG] = 1000 * (time.perf_counter() - wall) / len(TEST_PATHS)
    print(f"{HOG}: test {ALL_CORE_MS[HOG]:.0f} ms/img on {CORES} cores (images run in parallel)")
else:
    pred_dir = C.OUT / "preds" / "test"
    PREDS[HOG] = {s: load_predictions(pred_dir / f"{s}.txt") for s in TEST_STEMS}
    ALL_CORE_MS[HOG] = np.nan
    print(f"{HOG}: loaded saved predictions from {pred_dir}")

## 2. Timing on one core

Every method runs on the **same images**, with numpy, OpenCV and PyTorch limited to **one thread**. This is the fairest speed comparison. All-core numbers from step 1 are shown alongside. Training times come from the training runs, on different hardware, and are labelled that way.

In [ ]:
rng = np.random.default_rng(0)
timing_idx = np.sort(rng.choice(len(TEST_PATHS), size=min(TIMING_IMAGES, len(TEST_PATHS)), replace=False))
timing_paths = [TEST_PATHS[i] for i in timing_idx]
ONE_CORE_MS = {}

torch.set_num_threads(1)
cv2.setNumThreads(1)
with threadpool_limits(limits=1):
    for name, path in YOLO_MODELS.items():
        model = YOLO(str(path))
        warm_up(model)
        _, sec = yolo_predict(model, timing_paths, [p.stem for p in timing_paths])
        ONE_CORE_MS[name] = 1000 * sec.mean()
        print(f"{name}: {ONE_CORE_MS[name]:.1f} ms/img (1 core, {len(sec)} images)")
    w, b = load_model()
    sec = []
    for p in timing_paths:
        t = time.perf_counter()
        detect(load_image(p), w, b)
        sec.append(time.perf_counter() - t)
    ONE_CORE_MS[HOG] = 1000 * np.mean(sec)
    print(f"{HOG}: {ONE_CORE_MS[HOG]:.0f} ms/img (1 core, {len(sec)} images)")
torch.set_num_threads(CORES)
cv2.setNumThreads(-1)

TRAINING = {  # from the training notebooks
    HOG: "9.6 min (12 CPU cores)",
    "YOLO26n 2-stage": "25.5 min (Colab T4 GPU, 20 frozen + 30 full epochs)",
    "YOLO26n no-freeze": "23.6 min (Colab T4 GPU, 50 epochs)",
}
timing = pd.DataFrame({
    "training time": {k: v for k, v in TRAINING.items() if k in ONE_CORE_MS},
    "test ms / image (1 core)": ONE_CORE_MS,
    f"test ms / image ({CORES} cores)": ALL_CORE_MS,
}).loc[[HOG, *YOLO_MODELS]]
timing["1-core speed-up vs HOG"] = timing.loc[HOG, "test ms / image (1 core)"] / timing["test ms / image (1 core)"]
display(timing.round(1))
timing.to_csv(OUT / "timing.csv")
print(f"Hardware for inference: {CPU}, {CORES} logical cores, CPU only.")

## 3. Accuracy with one evaluator

All methods are scored with `methods/hog_svm/evaluate.py` (VOC all-point AP). These YOLO numbers can differ slightly from Ultralytics' own `val()` output, which uses 101-point interpolation. Use **these** numbers in the report comparison, because every method was scored the same way.

In [ ]:
results = pd.DataFrame({name: {"operating threshold": OP_THR[name], **scores_table(PREDS[name], GT["test"], OP_THR[name])}
                        for name in [HOG, *YOLO_MODELS]}).T
display(results.round(3))
results.to_csv(OUT / "test_results.csv")

plt.figure(figsize=(6, 4.5))
for name in [HOG, *YOLO_MODELS]:
    ap, prec, rec = average_precision(PREDS[name], GT["test"], 0.5)
    plt.plot(rec, prec, label=f"{name} (AP {ap:.3f})")
plt.xlabel("recall"); plt.ylabel("precision"); plt.xlim(0, 1); plt.ylim(0, 1.02)
plt.title("Test precision-recall at IoU 0.5"); plt.grid(alpha=0.3); plt.legend()
savefig("pr_curves_test.png"); plt.show()

## 4. Missed birds compared with the labels

At each method's operating threshold, every labelled bird is either **found** (a detection overlaps it with IoU ≥ 0.5) or **missed**. Every detection that matches no bird is a **false box**.

In [ ]:
METHODS = [HOG, MAIN_YOLO]
rows = []
for s in TEST_STEMS:
    row = {"image": s, "labelled birds": len(GT["test"][s])}
    for name in METHODS:
        b, c = PREDS[name][s]
        keep = c > OP_THR[name]
        _, tp, found = match(b[keep], c[keep], GT["test"][s])
        row[f"{name}: found"] = int(found.sum())
        row[f"{name}: missed"] = int((~found).sum())
        row[f"{name}: false boxes"] = int((~tp).sum())
    rows.append(row)
per_image = pd.DataFrame(rows).set_index("image")
per_image.to_csv(OUT / "per_image_missed.csv")

total = per_image["labelled birds"].sum()
summary = pd.DataFrame({name: {
    "labelled birds": total,
    "found": per_image[f"{name}: found"].sum(),
    "missed": per_image[f"{name}: missed"].sum(),
    "missed %": 100 * per_image[f"{name}: missed"].sum() / total,
    "false boxes": per_image[f"{name}: false boxes"].sum(),
    "images with every bird found": int(((per_image[f"{name}: missed"] == 0) & (per_image["labelled birds"] > 0)).sum()),
} for name in METHODS}).T
display(summary.round(1))
summary.to_csv(OUT / "missed_summary.csv")

for name in METHODS:
    print(f"\nImages where {name} missed the most birds:")
    display(per_image.sort_values(f"{name}: missed", ascending=False).head(8))

fig, ax = plt.subplots(figsize=(5.5, 5))
lim = per_image["labelled birds"].max() + 5
for name, marker in zip(METHODS, ["x", "o"]):
    detected = per_image[f"{name}: found"] + per_image[f"{name}: false boxes"]
    ax.scatter(per_image["labelled birds"], detected, s=16, alpha=0.6, marker=marker, label=name)
ax.plot([0, lim], [0, lim], "k--", lw=1, label="perfect count")
ax.set(xlabel="labelled birds in image", ylabel="birds detected", title="Counting on the test set", xlim=(0, lim), ylim=(0, lim))
ax.legend(); ax.grid(alpha=0.3)
savefig("count_scatter.png"); plt.show()

In [ ]:
def draw_errors(stem, ax, name):
    # Green = found, red = false box, blue dashed = missed bird.
    img = cv2.cvtColor(load_image(C.DATA / "images" / "test" / f"{stem}.jpg"), cv2.COLOR_BGR2RGB)
    b, c = PREDS[name][stem]
    keep = c > OP_THR[name]
    boxes, tp, found = match(b[keep], c[keep], GT["test"][stem])
    ax.imshow(img)
    for (x1, y1, x2, y2), ok in zip(boxes, tp):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=1.2, edgecolor="lime" if ok else "red"))
    for (x1, y1, x2, y2) in GT["test"][stem][~found]:
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=1.2, edgecolor="deepskyblue", linestyle="--"))
    ax.set_title(f"{name}: {found.sum()}/{len(found)} found, {(~found).sum()} missed, {(~tp).sum()} false", fontsize=9)
    ax.axis("off")


examples = {
    "most missed by YOLO": per_image[f"{MAIN_YOLO}: missed"].idxmax(),
    "most missed by HOG": per_image[f"{HOG}: missed"].idxmax(),
    "most false boxes from HOG": per_image[f"{HOG}: false boxes"].idxmax(),
}
fig, axes = plt.subplots(len(examples), 2, figsize=(10, 5 * len(examples)))
for row, (why, stem) in enumerate(examples.items()):
    for col, name in enumerate(METHODS):
        draw_errors(stem, axes[row, col], name)
    axes[row, 0].text(0, -0.08, f"{stem} ({why})", transform=axes[row, 0].transAxes, fontsize=9)
plt.suptitle("Green: found   Red: false box   Blue dashed: missed bird", y=1.0)
plt.tight_layout()
savefig("missed_birds_examples.png"); plt.show()

## 5. Big vs small birds

Bird size is the square root of the labelled box area in pixels. The median bird is about 42 px. Recall is the share of birds in each size group that were found.

In [ ]:
SIZE_BINS = [(0, 16, "< 16 px"), (16, 32, "16-32 px"), (32, 64, "32-64 px"), (64, 1e9, "> 64 px")]
size_rows = {lab: {"birds": 0, **{name: 0 for name in METHODS}} for *_, lab in SIZE_BINS}
missed_sizes = {name: [] for name in METHODS}
for s in TEST_STEMS:
    gt = GT["test"][s]
    if not len(gt):
        continue
    side = np.sqrt((gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1]))
    found = {}
    for name in METHODS:
        b, c = PREDS[name][s]
        keep = c > OP_THR[name]
        found[name] = match(b[keep], c[keep], gt)[2]
        missed_sizes[name] += list(side[~found[name]])
    for lo, hi, lab in SIZE_BINS:
        inside = (side >= lo) & (side < hi)
        size_rows[lab]["birds"] += int(inside.sum())
        for name in METHODS:
            size_rows[lab][name] += int(found[name][inside].sum())

size_table = pd.DataFrame(size_rows).T
for name in METHODS:
    size_table[f"{name} recall"] = size_table[name] / size_table["birds"].replace(0, np.nan)
size_table = size_table[["birds", *[f"{n} recall" for n in METHODS]]]
display(size_table.round(3))
size_table.to_csv(OUT / "recall_by_size.csv")

ax = size_table[[f"{n} recall" for n in METHODS]].plot.bar(figsize=(7, 4), rot=0)
ax.set(ylabel="recall at operating threshold", ylim=(0, 1), title="Recall by bird size (test)")
for i, n in enumerate(size_table["birds"]):
    ax.text(i, 0.97, f"n={int(n)}", ha="center", fontsize=8)
ax.grid(axis="y", alpha=0.3)
savefig("recall_by_size.png"); plt.show()

## 6. Bright vs dark images

Brightness is the mean grey level of the image (0 = black, 255 = white). The dark, medium and bright groups are split at the **33rd and 67th percentiles of the training images**, so the test set doesn't define its own groups.

In [ ]:
def brightness(path):
    return float(cv2.cvtColor(load_image(path), cv2.COLOR_BGR2GRAY).mean())

train_bright = np.array([brightness(p) for p in image_paths("train")])
low, high = np.percentile(train_bright, [33.3, 66.7])
test_bright = pd.Series({p.stem: brightness(p) for p in TEST_PATHS})
groups = pd.cut(test_bright, [-1, low, high, 256], labels=["dark", "medium", "bright"])
print(f"Group edges from training images: dark < {low:.0f} <= medium < {high:.0f} <= bright")

bright_rows = []
for g in ["dark", "medium", "bright"]:
    stems = list(groups[groups == g].index)
    for name in METHODS:
        r = scores_table(PREDS[name], GT["test"], OP_THR[name], stems)
        bright_rows.append({"brightness": g, "method": name, **r})
bright_table = pd.DataFrame(bright_rows).set_index(["brightness", "method"])
display(bright_table[["images", "birds", "mAP@0.5", "precision", "recall", "false boxes / image"]].round(3))
bright_table.to_csv(OUT / "accuracy_by_brightness.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bright_table["mAP@0.5"].unstack().loc[["dark", "medium", "bright"], METHODS].plot.bar(ax=axes[0], rot=0)
axes[0].set(title="mAP@0.5 by image brightness (test)", ylabel="mAP@0.5", ylim=(0, 1))
axes[1].hist(train_bright, bins=40, alpha=0.5, density=True, label="train")
axes[1].hist(test_bright, bins=40, alpha=0.5, density=True, label="test")
for x in (low, high):
    axes[1].axvline(x, color="k", ls="--")
axes[1].set(title="Image brightness", xlabel="mean grey level"); axes[1].legend()
for a in axes:
    a.grid(alpha=0.3)
plt.tight_layout()
savefig("accuracy_by_brightness.png"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, stem, label in [(axes[0], test_bright.idxmin(), "darkest"), (axes[1], test_bright.idxmax(), "brightest")]:
    ax.imshow(cv2.cvtColor(load_image(C.DATA / "images" / "test" / f"{stem}.jpg"), cv2.COLOR_BGR2RGB))
    ax.set_title(f"{label} test image: {stem} (mean {test_bright[stem]:.0f})", fontsize=9); ax.axis("off")
savefig("brightness_extremes.png"); plt.show()

## 7. Optional: simulated lighting changes

Each test image's pixel values are multiplied by a factor, then clipped to 0–255. A factor of 0.4 is a dim scene, 1.0 is the original, and 1.6 is over-exposed. The models are not retrained. This tests robustness, which counts toward the higher marks band.

YOLO takes a minute or two. HOG re-runs the full test set once per factor, so it's off by default (`RUN_SYNTHETIC_HOG`).

In [ ]:
FACTORS = [0.4, 0.6, 0.8, 1.0, 1.3, 1.6]

def adjust(img, factor):
    return np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)

synthetic = {}
if RUN_SYNTHETIC_YOLO:
    model = YOLO(str(YOLO_MODELS[MAIN_YOLO]))
    warm_up(model)
    for f in FACTORS:
        imgs = [adjust(load_image(p), f) for p in TEST_PATHS]
        preds, _ = yolo_predict(model, imgs, TEST_STEMS)
        synthetic[(MAIN_YOLO, f)] = average_precision(preds, GT["test"], 0.5)[0]
        print(f"{MAIN_YOLO} x{f}: mAP@0.5 = {synthetic[(MAIN_YOLO, f)]:.3f}")

if RUN_SYNTHETIC_HOG:
    from joblib import Parallel, delayed
    w, b = load_model()
    for f in FACTORS:
        out = Parallel(n_jobs=C.N_JOBS)(delayed(detect)(adjust(load_image(p), f), w, b) for p in TEST_PATHS)
        preds = dict(zip(TEST_STEMS, out))
        synthetic[(HOG, f)] = average_precision(preds, GT["test"], 0.5)[0]
        print(f"{HOG} x{f}: mAP@0.5 = {synthetic[(HOG, f)]:.3f}")

if synthetic:
    syn = pd.Series(synthetic).unstack(0)
    display(syn.round(3))
    syn.to_csv(OUT / "synthetic_brightness.csv")
    ax = syn.plot(marker="o", figsize=(6, 4))
    ax.set(xlabel="brightness factor (1.0 = original)", ylabel="test mAP@0.5", ylim=(0, 1),
           title="Robustness to simulated lighting")
    ax.grid(alpha=0.3)
    savefig("synthetic_brightness.png"); plt.show()

## 8. What to take into the report

- **Table II (main results):** `test_results.csv` plus `timing.csv`. Report the CPU model printed in step 0 as the inference hardware.
- **Fig. PR curves:** `figures/pr_curves_test.png`.
- **Failure analysis:** `missed_summary.csv`, `figures/missed_birds_examples.png`, `figures/count_scatter.png`.
- **Bird size:** `recall_by_size.csv`, `figures/recall_by_size.png`.
- **Brightness:** `accuracy_by_brightness.csv`, `figures/accuracy_by_brightness.png` and, if step 7 was run, `figures/synthetic_brightness.png`.
- Say in the report that both methods were timed on the same CPU and scored with the same evaluation code, and that YOLO's operating threshold was chosen on validation.